In [1]:
import rasterio
import numpy as np
import pandas as pd
from rasterio.windows import Window
from rasterio.transform import rowcol
from pyproj import Transformer, CRS
import pickle

In [2]:
dte_path = '../00_raw_data/dte_visibility/AVGVISIB_75S_120M_201608_EARTH.tiff'

with rasterio.open(dte_path) as src:
    print(f"CRS:        {src.crs}")
    print(f"Resolution: {src.res} m/px")
    print(f"Bounds:     {src.bounds}")
    print(f"NoData:     {src.nodata}")

    arr_check = src.read(1).astype(float)
    arr_check[arr_check == src.nodata] = np.nan
    total_ts_dte = float(np.nanmax(arr_check))
    del arr_check

    print(f"\nTotal timesteps: {total_ts_dte:.0f}")
    px_size_dte = src.res[0]

CRS:        PROJCS["Moon2000_spole",GEOGCS["GCS_Moon",DATUM["D_Moon_2000",SPHEROID["Moon_2000_IAU_IAG",1737400,0]],PRIMEM["Reference_Meridian",0],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]]],PROJECTION["Polar_Stereographic"],PARAMETER["latitude_of_origin",-90],PARAMETER["central_meridian",0],PARAMETER["false_easting",0],PARAMETER["false_northing",0],UNIT["metre",1],AXIS["Easting",NORTH],AXIS["Northing",NORTH]]
Resolution: (120.0, 120.0) m/px
Bounds:     BoundingBox(left=-457440.0, bottom=-457440.0, right=457440.0, top=457440.0)
NoData:     -32768.0


C:\Users\Aska\AppData\Local\Temp\ipykernel_25616\1473428680.py:9: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr_check = src.read(1).astype(float)



Total timesteps: 25000


In [3]:
sites_df = pd.read_csv('../01_cleaned_data/sites_df.csv')

moon_gcs = CRS.from_proj4('+proj=longlat +R=1737400 +no_defs')

with rasterio.open(dte_path) as src:
    dte_crs = src.crs

transformer_dte = Transformer.from_crs(moon_gcs, dte_crs, always_xy=True)

sites_df['X_dte'], sites_df['Y_dte'] = transformer_dte.transform(
    sites_df['Lon'].values,
    sites_df['Lat'].values
)

In [4]:
# ref: Gracy & Lee (LPSC 2024) https://www.hou.usra.edu/meetings/lpsc2024/pdf/1695.pdf

def region_half_window_px(area_km2, perimeter_km, px_size_m):
    half_perim = perimeter_km / 2
    disc       = half_perim**2 - 4 * area_km2
    long_side  = (half_perim + np.sqrt(max(disc, 0))) / 2 if disc >= 0 \
                 else np.sqrt(area_km2)
    return int((long_side * 1000 / 2) / px_size_m) + 1

ez_radius_px_dte = 2000 // px_size_dte

with rasterio.open(dte_path) as src:
    for _, row in sites_df.iterrows():
        search_ws = region_half_window_px(row['Area_km2'], row['Perimeter_km'],
                                           px_size_dte)
        buf = ez_radius_px_dte

        py, px_c = rowcol(src.transform, row['X_dte'], row['Y_dte'])

        r0 = max(0, py   - search_ws - buf)
        r1 = min(src.height, py + search_ws + buf)
        c0 = max(0, px_c - search_ws - buf)
        c1 = min(src.width,  px_c + search_ws + buf)

        win = Window(c0, r0, c1-c0, r1-r0)
        arr = src.read(1, window=win).astype(float)
        arr[arr == src.nodata] = np.nan

        pct_arr = (arr / total_ts_dte) * 100

        window_transform = rasterio.windows.transform(win, src.transform)

        site_key  = row['site_key']
        grid_file = f'../01_cleaned_data/site_grids/{site_key}_grids.pkl'

        with open(grid_file, 'rb') as f:
            g = pickle.load(f)

        g['dte_pct']       = pct_arr
        g['dte_transform'] = window_transform
        g['px_size_dte']   = px_size_dte

        with open(grid_file, 'wb') as f:
            pickle.dump(g, f)

        print(f"{row['Site']:25s}  DTE grid: {pct_arr.shape}  "
              f"range: {np.nanmin(pct_arr):.1f}-{np.nanmax(pct_arr):.1f}%")

print("\nDTE grids saved.")

C:\Users\Aska\AppData\Local\Temp\ipykernel_25616\3726748358.py:26: DeprecationWarning: Setting the shape on a NumPy array has been deprecated in NumPy 2.5.
As an alternative, you can create a new view using np.reshape (with copy=False if needed).
  arr = src.read(1, window=win).astype(float)


Nobile Rim 2               DTE grid: (200, 200)  range: 0.0-88.0%
Mons Mouton                DTE grid: (166, 166)  range: 0.0-100.0%
Malapert Massif            DTE grid: (208, 208)  range: 0.0-100.0%
de Gerlache Rim 2          DTE grid: (200, 200)  range: 0.0-61.9%
Mons Mouton Plateau        DTE grid: (672, 672)  range: 0.0-100.0%
Slater Plain               DTE grid: (200, 200)  range: 0.0-50.0%
Peak near Cabeus B         DTE grid: (200, 200)  range: 0.0-99.5%
Nobile Rim 1               DTE grid: (200, 200)  range: 0.0-100.0%
Haworth                    DTE grid: (282, 282)  range: 0.0-82.2%

DTE grids saved.
